In [1]:
# Célula 1: Configuração do Laboratório
import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import pandas_ta as ta
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
import optuna
from functools import partial
from pprint import pprint
from datetime import datetime, timedelta
import warnings
import sys
import os

# Adiciona a pasta raiz do projeto ao caminho do Python.
sys.path.append(os.path.abspath('..'))

# --- Configurações Visuais e de Ambiente ---
warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Laboratório configurado.")

# --- Variáveis de Controle Globais ---
ATIVO_TESTE = "GBPUSD"
DIAS_TESTE = 2000
CAPITAL_INICIAL = 10000.0
RISCO_POR_TRADE_PCT = 0.01

# ### INÍCIO DA CORREÇÃO DEFINITIVA ###
# Importamos TODOS os nossos módulos do quant_engine aqui, de uma só vez,
# garantindo que estejam disponíveis para todo o notebook.
from robots.quant_engine import data_processor, portfolio_analyzer, ml_trainer
# ### FIM DA CORREÇÃO DEFINITIVA ###
from robots.quant_engine.strategies import lh_agressive, lh_conservative

print("Módulos do projeto importados com sucesso.")

Laboratório configurado.
Módulos do projeto importados com sucesso.


In [2]:
# Célula 2: Coleta de Dados de Mercado
print(f"--- Coletando dados para: {ATIVO_TESTE} ---")
df_mercado = data_processor.prepare_data_for_portfolio([ATIVO_TESTE], dias=DIAS_TESTE)

if df_mercado and ATIVO_TESTE in df_mercado:
    df_mercado_ativo = df_mercado[ATIVO_TESTE]
    print(f"Dados para {ATIVO_TESTE} carregados com sucesso. Shape: {df_mercado_ativo.shape}")
    display(df_mercado_ativo.tail())
else:
    raise ValueError("Falha ao carregar os dados de mercado.")

--- Coletando dados para: GBPUSD ---
  [DATA_PROC]: Iniciando coleta de dados para o portfólio: ['GBPUSD']...
    - Coletando para: GBPUSD
    - Preparação concluída para GBPUSD. Total de 135846 candles.
Dados para GBPUSD carregados com sucesso. Shape: (135846, 12)


,time,open,high,low,close,volume,spread,real_volume,nivel_liquidez_sup,nivel_liquidez_inf,rsi,atr
135841,2025-07-22 14:45:00,1.34788,1.34881,1.34756,1.34860,1072,8,0,1.35102,1.34014,54.060796,0.000732
135842,2025-07-22 15:00:00,1.34860,1.34895,1.34845,1.34859,889,8,0,1.35102,1.34014,53.956954,0.000715
135843,2025-07-22 15:15:00,1.34860,1.34863,1.34760,1.34766,918,8,0,1.35102,1.34014,45.251538,0.000738
135844,2025-07-22 15:30:00,1.34765,1.34811,1.34749,1.34792,878,8,0,1.35102,1.34014,47.787772,0.000730
135845,2025-07-22 15:45:00,1.34793,1.34863,1.34768,1.34805,1213,8,0,1.35102,1.34014,49.058476,0.000745


In [3]:
# Célula 3: Definição da Função Objetivo para o Optuna (Corrigida)

def objective(trial, strategy_module, df_mercado_base, capital_inicial, risco_pct, ml_trainer_module, portfolio_analyzer_module):
    """
    Função objetivo universal. Testa uma combinação de hiperparâmetros para uma
    dada estratégia e retorna o score de performance (Retorno/MDD).
    """
    # 1. Sugere os parâmetros a serem testados neste "trial"
    params = {
        'Risco_Retorno': trial.suggest_float('Risco_Retorno', 1.0, 3.5, step=0.1),
        'Max_Candles_Hold': trial.suggest_int('Max_Candles_Hold', 20, 120, step=4),
        'Stop_Loss_ATR_Mult': trial.suggest_float('Stop_Loss_ATR_Mult', 0.5, 3.0, step=0.1),
        'EMA_Curta_Periodo': 21,
        'EMA_Longa_Periodo': 200
    }
    
    try:
        # 2. Geração de Sinais e Features
        df_mercado_featured = strategy_module.add_indicators(df_mercado_base, params)
        alertas = strategy_module.generate_alerts(df_mercado_featured, params)
        if alertas.empty: return -1.0
        
        dataset_para_ia = strategy_module.define_decision_points_and_features(alertas, df_mercado_featured, params)
        if dataset_para_ia.empty: return -1.0

        # 3. Rotulagem
        dataset_rotulado = portfolio_analyzer_module.label_trades(dataset_para_ia, df_mercado_featured, params)
        if 'target' not in dataset_rotulado.columns:
            return -1.0

        # 4. Simulação Walk-Forward
        features = [col for col in dataset_rotulado.columns if col.startswith('feature_')]
        X = dataset_rotulado[features]
        y = dataset_rotulado['target']
        
        if len(X) < 12: return 0.0

        tscv = TimeSeriesSplit(n_splits=5)
        all_tested_trades_df = pd.DataFrame()
        
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
            df_treino = dataset_rotulado.iloc[train_idx]
            df_teste = dataset_rotulado.iloc[test_idx].copy()
            
            # Chama a função a partir do módulo que foi passado como argumento.
            model_fold, features_fold = ml_trainer_module.train_model_on_past_data(df_treino)
            if model_fold is None: continue

            X_teste = df_teste[features_fold]
            if X_teste.empty: continue
            
            df_teste['previsao_ia'] = model_fold.predict(X_teste)
            trades_aprovados_neste_fold = df_teste[df_teste['previsao_ia'] == 1]
            all_tested_trades_df = pd.concat([all_tested_trades_df, trades_aprovados_neste_fold])

        if all_tested_trades_df.empty:
            return 0.0

        # 5. Simulação Financeira Final
        # Esta lógica precisa ser movida para dentro do portfolio_analyzer
        final_stats = portfolio_analyzer_module.run_financial_simulation(
            all_tested_trades_df, capital_inicial, DIAS_TESTE
        )

        # 6. Cálculo do Score Final
        net_profit = final_stats.get("Lucro Líquido ($)", 0)
        max_dd = final_stats.get("Drawdown Máximo [%]", 100) / 100
        total_trades = final_stats.get("Nº de Trades Total", 0)
        
        score = (net_profit / capital_inicial) / max_dd if max_dd > 0.001 else 0
        return score * np.log1p(total_trades)

    except Exception as e:
        return -1.0

print("Motor de Otimização 'objective' (v2) definido com sucesso.")

Motor de Otimização 'objective' (v2) definido com sucesso.


In [4]:
# Célula 4: A Forja de Alfas - Execução das Otimizações (Corrigida)

resultados_finais_otimizacao = {}

estrategias_para_otimizar = {
    "LH Agressive": lh_agressive.LhAgressive,
    "LH Conservative": lh_conservative.LhConservative
}

for nome_estrategia, modulo_estrategia in estrategias_para_otimizar.items():
    print("\n" + "#"*80)
    print(f"             INICIANDO OTIMIZAÇÃO PARA: {nome_estrategia} (ATIVO: {ATIVO_TESTE})")
    print("#"*80)

    study = optuna.create_study(direction='maximize')

    # ### INÍCIO DA CORREÇÃO DEFINITIVA ###
    # Passamos os módulos 'ml_trainer' e 'portfolio_analyzer' como argumentos fixos.
    objective_with_data = partial(
        objective,
        strategy_module=modulo_estrategia,
        df_mercado_base=df_mercado_ativo,
        capital_inicial=CAPITAL_INICIAL,
        risco_pct=RISCO_POR_TRADE_PCT,
        ml_trainer_module=ml_trainer,
        portfolio_analyzer_module=portfolio_analyzer
    )
    # ### FIM DA CORREÇÃO DEFINITIVA ###

    N_TRIALS = 100
    print(f"\nIniciando otimização com {N_TRIALS} tentativas...")
    
    study.optimize(objective_with_data, n_trials=N_TRIALS, show_progress_bar=True)
    
    print(f"\nOtimização para {nome_estrategia} concluída.")
    
    resultados_finais_otimizacao[nome_estrategia] = {
        "Melhor Score": study.best_value,
        "Melhores Parâmetros": study.best_params
    }

# --- SUMÁRIO FINAL COMPARATIVO ---
print("\n\n" + "="*80)
print("                    SUMÁRIO FINAL DAS OTIMIZAÇÕES")
print("="*80)

for nome, resultado in resultados_finais_otimizacao.items():
    print(f"\n--- ESTRATÉGIA: {nome} ---")
    print(f"  Melhor Score (Retorno/MDD Ajustado): {resultado['Melhor Score']:.2f}")
    print("  DNA Otimizado:")
    pprint(resultado['Melhores Parâmetros'])
    print("-" * 50)

print("\n" + "="*80)


################################################################################
             INICIANDO OTIMIZAÇÃO PARA: LH Agressive (ATIVO: GBPUSD)
################################################################################

Iniciando otimização com 100 tentativas...


  0%|          | 0/100 [00:00<?, ?it/s]

    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 19.81%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 32.87%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 21.27%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 21.43%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 19.32%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 42.69%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 12.91%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 28.98%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 24.19%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 41.64%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 18.10%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 27.35%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 35.47%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 20.45%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 10.63%
    [ANALYZER]: Rotulagem completa. Taxa

  0%|          | 0/100 [00:00<?, ?it/s]

    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 17.60%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 17.01%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 8.80%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 20.23%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 35.48%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 0.29%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 1.17%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 10.85%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 43.40%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 17.89%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 12.32%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 0.59%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 0.59%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 19.94%
    [ANALYZER]: Rotulagem completa. Taxa de acerto base: 4.99%
    [ANALYZER]: Rotulagem completa. Taxa de ac